# BiciTierra Market · Explorar dades abans de predir

Dades sintètiques: 66 mesos i 18 sèries comercials. Objectiu: predir unitats d’un mes abans que comence. Llegiu el [diccionari](../dades/DICCIONARI_DADES.md).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('..') / 'dades'
sales = pd.read_csv(DATA / 'sales_history.csv')
sales['month'] = pd.to_datetime(sales['month'], format='%Y-%m')
KEY = ['channel', 'product_line', 'region']
sales = sales.sort_values(KEY + ['month']).reset_index(drop=True)
assert not sales.duplicated(['month'] + KEY).any()
sales.head()
customers = pd.read_csv(DATA / 'customers.csv')
print('Files de vendes:', len(sales), 'Clients:', len(customers))

## 1. Estructura, cobertura i qualitat

Cada fila de vendes és un mes, canal, producte i regió. Comproveu la cobertura abans de crear retards.

In [ ]:
coverage = sales.groupby(KEY)['month'].agg(['min', 'max', 'nunique'])
print(coverage)
print(sales.isna().sum())
print(customers.isna().sum())
print('Duplicats de clau:', sales.duplicated(['month'] + KEY).sum())

## 2. Separar el temps abans de prendre decisions de model

Exploreu patrons amb entrenament. Reserveu 2025 per comparar decisions i 2026 per a la prova final. No és una partició aleatòria.

In [ ]:
train = sales[sales['month'] < '2025-01-01'].copy()
validation = sales[(sales['month'] >= '2025-01-01') & (sales['month'] < '2026-01-01')]
test = sales[sales['month'] >= '2026-01-01']
print({'train': len(train), 'validation': len(validation), 'test': len(test)})

## 3. Estacionalitat i diferències comercials

Compareu mesos, anys i productes. Una correlació en dades simulades no demostra una causa al món real.

In [ ]:
train['month_number'] = train['month'].dt.month
seasonality = train.pivot_table(index='month_number', columns='product_line', values='sales_units', aggfunc='mean').round(1)
seasonality

In [ ]:
train.groupby(['product_line', 'channel', 'region'])['sales_units'].agg(['mean','std','min','max']).round(2)

In [ ]:
train.assign(year=train['month'].dt.year).groupby(['year','product_line'])['sales_units'].sum().unstack()

## 4. Estoc i informació disponible

`stockout`, estoc final, sessions i ingressos només es coneixen al tancament. Són útils per analitzar, però no com a predictors del mateix mes.

In [ ]:
print('Vendes superiors a estoc:', (sales['sales_units'] > sales['stock_units']).sum())
print('Estoc final incoherent:', (sales['closing_stock_units'] != sales['stock_units'] - sales['sales_units']).sum())
train.groupby('product_line')['stockout'].mean().rename('proporcio_ruptures')

## 5. Clients: un problema separat

No uniu clients amb vendes per ciutat, canal o ordre de files. Són mostres independents. La fotografia de clients és del 30/06/2026.

In [ ]:
customers[['annual_spend','avg_order_value','orders_last_12m','returns_ratio','days_since_last_order']].describe().round(2)

## Entrega

Escriviu una pregunta de predicció, dues hipòtesis sobre entrenament, el tractament proposat dels nuls i una llista de predictors disponibles abans del mes. Expliqueu per què no es pot usar `revenue` per predir `sales_units` del mateix mes.